# 采样轴、单位与双向位置

本 Notebook 将同一品种的实际分钟数据转换为采样轴，利用初始训练期确定单位 $a$，再从指定分界 $t$ 向过去和未来生成采样位置。位置包含源分钟、分钟内比例及采样时间，供各合约共用。

## 目录

- [每分钟增量与初始单位](#axis-and-unit)
- [两种采样规则](#sampling-rules)
  - [分钟内插值：`log_interpolate`](#log-interpolation)
  - [分钟末清零：`minute_close_reset`](#minute-close-reset)
- [指定分界与滚动分界](#fixed-and-rolling-boundaries)
- [合成分钟位置 demo](#synthetic-position-demo)
- [真实分钟数据的位置 demo](#actual-position-demo)
- [48 组位置数据 demo](#saved-位置-demos)

输入是[主力连续复权 Notebook](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b02_MainContinuousAdjustment/c01_MainContinuousAdjustment.ipynb)生成的全部合约分钟数据。`bar_at` 表示分钟末端，`trading_date` 表示所属交易日；时间按 Asia/Shanghai 解释。午休、隔夜和跨交易日的空档不补分钟，相邻实际记录直接衔接。

采样需要区分四个量：

| 量 | 作用 |
| --- | --- |
| $N$ | 用于校准的候选分钟数，demo 使用 10、15、30、60。 |
| $a$ | 一个采样间隔对应的轴增量，由初始训练期的分钟平均增量乘 $N$ 得到。 |
| $t$ | 本次采样的训练末端，已知该分钟的 close；过去属于训练侧，未来属于测试侧。 |
| $K$ | 每侧最多返回的采样点数；训练侧最后的 $t$ 点也占一个名额。 |

$a$ 校准后固定。改变 $t$ 会改变这次训练、测试采样序列的分界，单位与输入复权价格保持固定。

以下先定义采样轴和单位，再说明双向采样规则，最后演示合成分钟和真实数据的位置计算。各合约价格的生成、保存及图表见[采样执行 Notebook](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c02_SamplingExecution.ipynb)。


In [1]:
import pathlib
import sys

# 定位项目根并启用仓库配置、研究方法的正常导入。
project_markers = ['.git', '.env', 'config/settings.py']
current_path = pathlib.Path.cwd()
for candidate_root in [current_path, *current_path.parents]:
    if all(((candidate_root / marker).exists() for marker in project_markers)):
        sys.path.insert(0, str(candidate_root))
        from config.settings import settings
        break
else:
    raise RuntimeError('无法定位项目根，请在包含 .git、.env、config/settings.py 的项目中运行')


In [2]:
from __future__ import annotations

import numpy as np
import pandas as pd
import pyarrow as pa
from numba import njit, prange
from config.data_contracts import FUTURES_MINUTE_SCHEMA

SAMPLING_POSITIONS_VERSION = '0.1.0'

# 定义每个分界的采样位置；各合约共用这些时间与比例。
SAMPLING_POSITION_SCHEMA = pa.schema([
    pa.field('sample_id', pa.int64(), nullable=False),
    pa.field('t', FUTURES_MINUTE_SCHEMA.field('bar_at').type, nullable=False),  # 已知 close 的训练末端
    pa.field('sample_side', pa.string(), nullable=False),
    pa.field('sample_number', pa.int64(), nullable=False),  # 该侧从早到晚的序号
    pa.field('source_minute_index', pa.int64(), nullable=False),  # 实际分钟轴中的行号
    pa.field('sample_at', FUTURES_MINUTE_SCHEMA.field('bar_at').type, nullable=False),
    FUTURES_MINUTE_SCHEMA.field('trading_date'),
    FUTURES_MINUTE_SCHEMA.field('bar_at'),
    pa.field('main_contract_code', pa.string(), nullable=False),
    pa.field('interpolation_fraction', pa.float64(), nullable=False),  # 0 为 open，1 为 close
], metadata={b'method_name': b'sampling_positions',
             b'method_version': SAMPLING_POSITIONS_VERSION.encode(),
             b'timezone': b'Asia/Shanghai', b'primary_key': b't,sample_side,sample_number',
             b'grain_zh': '每个分界时点的每个采样位置一行'.encode()})

from datetime import date, datetime

SAMPLING_AXIS_VERSION = '0.2.0'
SAMPLING_TIMEZONE = 'Asia/Shanghai'

# 每个实际分钟保留一行轴增量，数量和主力价格各自按分钟对齐。
SAMPLING_AXIS_SCHEMA = pa.schema([
    FUTURES_MINUTE_SCHEMA.field('exchange_code'), FUTURES_MINUTE_SCHEMA.field('underlying_code'),
    FUTURES_MINUTE_SCHEMA.field('trading_date'), FUTURES_MINUTE_SCHEMA.field('bar_at'),
    pa.field('main_contract_code', pa.string(), nullable=False),
    pa.field('gk_sigma', pa.float64()),  # 成交额轴留空；GK×成交额轴使用主力分钟波动率
    pa.field('sampling_increment', pa.float64(), nullable=False),  # 该分钟推进的采样刻度
], metadata={
    b'method_name': b'sampling_axis', b'method_version': SAMPLING_AXIS_VERSION.encode(),
    b'timezone': SAMPLING_TIMEZONE.encode(), b'grain_zh': '每个实际分钟一行'.encode(),
})

# 读取身份、成交额和复权 OHLC；简单字段名按用途分组。
SAMPLING_INPUT_COLUMNS = [
    'exchange_code', 'underlying_code', 'contract_code', 'trading_date', 'bar_at',
    'main_contract_code', 'is_main', 'money',
    'adjusted_open', 'adjusted_high', 'adjusted_low', 'adjusted_close',
]


In [3]:
# 中文名称只用于表格展示副本，不改写计算字段或文件。
POSITION_COLUMN_LABELS = {
    field.name: f"{field.name}（{field.metadata[b'field_name_zh'].decode('utf8')}）"
    for field in FUTURES_MINUTE_SCHEMA
}

POSITION_COLUMN_LABELS.update({
    'sample_id': 'sample_id（采样编号）', 't': 't（训练期末端）',
    'sample_side': 'sample_side（采样所属区间）', 'sample_number': 'sample_number（区间内采样序号）',
    'source_minute_index': 'source_minute_index（源实际分钟位置）',
    'sample_at': 'sample_at（分钟内采样时间）', 'main_contract_code': 'main_contract_code（当分钟主力合约）',
    'interpolation_fraction': 'interpolation_fraction（源分钟内插值比例）',
    'sampling_increment': 'sampling_increment（每分钟采样轴增量）',
    'gk_sigma': 'gk_sigma（主力分钟 GK 波动率代理量）',
})



In [1]:
# 展示本 Notebook 输出字段、类型及空值约束。
display(pd.DataFrame([
    {'数据结构': name, '字段': POSITION_COLUMN_LABELS[field.name], 'Arrow 类型': str(field.type), '允许空值': field.nullable}

    for name, schema in (('sampling_axis', SAMPLING_AXIS_SCHEMA), ('sampling_position', SAMPLING_POSITION_SCHEMA)) for field in schema
]))

NameError: name 'pd' is not defined

## 每分钟增量与初始单位

<a id="axis-and-unit"></a>

`generate_sampling_column` 为每个实际分钟生成一行。记分钟 $i$ 的轴增量为 $w_i$，写入 `sampling_increment` 字段。

`quantity_scope` 决定成交额的取值范围：`all_contracts` 使用该分钟同品种全部合约的成交额合计；`main_contract` 使用该分钟主力合约的成交额。demo 使用 `all_contracts`；时间轴不使用成交额范围。轴由 `axis` 选择：

| `axis` | 分钟增量 $w_i$ |
| --- | --- |
| `time` | 每个实际分钟为 1；不补午休、隔夜或跨交易日的空档。 |
| `money` | 所选数量范围的成交额 $money_i$。 |
| `gk_money` | 主力分钟波动率乘成交额，即 $\sigma_i money_i$。 |

GK 波动率由函数计算。$O_i,H_i,L_i,C_i$ 分别为当分钟主力合约的复权 open、high、low、close：

$$
v_i=\frac12\left[\ln\left(\frac{H_i}{L_i}\right)\right]^2
-(2\ln2-1)\left[\ln\left(\frac{C_i}{O_i}\right)\right]^2,
\qquad \sigma_i=\sqrt{v_i}.
$$

这是单分钟的波动率代理量，不平滑、不年化。数量范围改变时，GK 的价格来源仍为当分钟主力。时间轴和成交额轴不需要计算 GK，其 `gk_sigma` 字段为空。

`calibrate_sampling_unit` 用 `test_start_date` 划定初始校准训练期：只取 `trading_date < test_start_date` 的实际分钟。记该集合为 $\mathcal T_0$，分钟数为 $M_0$，则

$$a=N\frac{\sum_{i\in\mathcal T_0}w_i}{M_0}.$$

时间轴的分钟增量恒为 1，校准得到 $a=N$；$N=10$ 表示每累计 10 个实际分钟到达一个刻度。成交额和 GK×成交额轴使用训练期平均每分钟增量的 $N$ 倍，实际采样间隔由各分钟增量决定。零增量分钟计入 $M_0$，但不推进采样刻度；$\sigma_i=0$ 时，GK×成交额的增量也为零，不设置最小权重。

所需成交额缺失或非有限时明确报错。构造 `gk_money` 时，主力 OHLC 必须有限且为正，GK 方差必须有限且非负，否则明确报错。训练期为空，或无法得到有限正单位，也明确报错。校准完成后，后续测试与滚动分界都使用这个 $a$。


In [5]:
def generate_sampling_column(adjusted_minute_df: pd.DataFrame, *, axis: str, quantity_scope: str = 'all_contracts') -> pd.DataFrame:
    """每实际分钟生成 sampling_increment；数量口径与采样算法分离。"""
    if axis not in ('time', 'money', 'gk_money'):
        raise ValueError('axis 必须为 time、money 或 gk_money')
    if quantity_scope not in ('all_contracts', 'main_contract'):
        raise ValueError('quantity_scope 必须为 all_contracts 或 main_contract')

    # 按实际分钟建立唯一轴索引，成交额与主力 OHLC 都对齐到这些分钟。
    axis_minute_df = (
        adjusted_minute_df.drop_duplicates('bar_at')
        [['exchange_code', 'underlying_code', 'trading_date', 'bar_at', 'main_contract_code']]
        .sort_values('bar_at').reset_index(drop=True)
    )


    # 时间轴只累计实际分钟；不读取成交额或 GK，休市空档不补行。
    if axis == 'time':
        axis_minute_df['gk_sigma'] = np.nan  # 时间轴不计算 GK
        axis_minute_df['sampling_increment'] = 1.0  # 每个实际分钟推进一个单位
        return pa.Table.from_pandas(axis_minute_df, schema=SAMPLING_AXIS_SCHEMA, preserve_index=False).to_pandas(types_mapper=pd.ArrowDtype)

    # 先检查所需成交额，再按分钟合计，避免 sum 隐藏缺失值。
    quantity_minute_df = adjusted_minute_df if quantity_scope == 'all_contracts' else adjusted_minute_df.loc[adjusted_minute_df['is_main']]
    quantity = quantity_minute_df['money'].to_numpy(dtype=float, na_value=np.nan)
    if not np.isfinite(quantity).all():
        raise ValueError('所需成交额 money 缺失或非有限')
    minute_money_totals = quantity_minute_df.groupby('bar_at', sort=False)['money'].sum()
    money = axis_minute_df['bar_at'].map(minute_money_totals).to_numpy(dtype=float, na_value=np.nan)
    if not np.isfinite(money).all():
        raise ValueError('所需成交额不可用：分钟缺主力或聚合非有限')
    sigma = np.full(len(axis_minute_df), np.nan)  # 成交额轴不计算 GK

    # GK 始终取当分钟主力 OHLC；数量范围只影响成交额。
    if axis == 'gk_money':
        main_minute_df = adjusted_minute_df.loc[adjusted_minute_df['is_main']].set_index('bar_at')
        main_prices_by_field = {
            name: axis_minute_df['bar_at'].map(main_minute_df['adjusted_' + name]).to_numpy(dtype=float, na_value=np.nan)
            for name in ('open', 'high', 'low', 'close')
        }
        if any(((~np.isfinite(values) | (values <= 0)).any() for values in main_prices_by_field.values())):
            raise ValueError('主力 OHLC 缺失或非正，无法计算 GK')
        main_logs_by_field = {name: np.log(values) for name, values in main_prices_by_field.items()}
        variance = (
            0.5 * (main_logs_by_field['high'] - main_logs_by_field['low']) ** 2  # 分钟振幅项
            - (2 * np.log(2) - 1) * (main_logs_by_field['close'] - main_logs_by_field['open']) ** 2  # 开收盘变动项
        )
        if (~np.isfinite(variance) | (variance < 0)).any():
            raise ValueError('主力 OHLC 无法得到有效的非负 GK 方差')
        sigma = np.sqrt(variance)  # 单分钟代理量，不平滑、不年化

    # 生成统一增量列，供校准和位置算法使用。
    increments = money if axis == 'money' else sigma * money
    if (~np.isfinite(increments) | (increments < 0)).any():
        raise ValueError('采样增量必须有限且非负')
    axis_minute_df['gk_sigma'] = sigma
    axis_minute_df['sampling_increment'] = increments
    return pa.Table.from_pandas(axis_minute_df, schema=SAMPLING_AXIS_SCHEMA, preserve_index=False).to_pandas(types_mapper=pd.ArrowDtype)

def calibrate_sampling_unit(axis_minute_df: pd.DataFrame, *, N: int, test_start_date: str | date) -> dict:
    """只用训练期的实际分钟平均增量确定单位 a。"""
    if isinstance(N, bool) or not isinstance(N, (int, np.integer)) or N <= 0:
        raise ValueError('N 必须为正整数')

    # 用交易日划定初始训练期；滚动 t 不参与单位校准。
    boundary = pd.Timestamp(test_start_date).date()
    training_axis_df = axis_minute_df.loc[axis_minute_df['trading_date'] < boundary]  # 分界交易日属于测试期
    if training_axis_df.empty:
        raise ValueError('训练期没有实际分钟，无法校准采样单位')
    increments = training_axis_df['sampling_increment'].to_numpy(dtype=float, na_value=np.nan)
    if (~np.isfinite(increments) | (increments < 0)).any():
        raise ValueError('训练采样增量必须有限且非负')

    # N 乘训练平均每分钟增量，零增量分钟仍计入分母。
    training_total_increment = float(np.sum(increments))
    sampling_unit = float(N * training_total_increment / len(training_axis_df))  # 后续所有分界复用的 a
    if not np.isfinite(sampling_unit) or sampling_unit <= 0:
        raise ValueError('训练期总增量为零或非有限，无法得到正采样单位')
    return {
        'N': int(N), 'a': sampling_unit,
        'training_minutes': len(training_axis_df), 'training_total_increment': training_total_increment,
        'training_start_date': training_axis_df['trading_date'].min().isoformat(),
        'training_end_date': training_axis_df['trading_date'].max().isoformat(), 'test_start_date': boundary.isoformat(),
    }


## 两种采样规则

<a id="sampling-rules"></a>

`prepare_sampling_axis` 接收分钟增量、固定单位 $a$ 和 `sampling_method`，为后续位置计算准备累计轴。休市空档不参与累计，也不触发重置；不足完整单位的尾量舍弃。

### 分钟内插值：`log_interpolate`

<a id="log-interpolation"></a>

记累计增量为 $S_i=\sum_{j\le i}w_j$，分界分钟末端的累计值为 $S_t$。过去的目标刻度依次为 $S_t-a,S_t-2a,\ldots$，未来的目标刻度为 $S_t+a,S_t+2a,\ldots$；训练侧另保留 $t$ 的 close 点。

目标刻度 $s$ 落在分钟 $i$ 内时，以

$$\theta=\frac{s-S_{i-1}}{w_i}$$

确定该分钟从 open 到 close 的位置。`interpolation_fraction` 保存 $\theta$，`sample_at` 为该分钟起点加 $\theta$ 分钟；`bar_at` 保留源分钟末端。目标恰好落在分钟末端时，$\theta=1$。一个分钟内经过几个完整目标刻度，就生成几个点。价格执行环节按同一比例对各合约的 log open、log close 作线性插值。

### 分钟末清零：`minute_close_reset`

<a id="minute-close-reset"></a>

未来方向从 $t$ 后的第一个实际分钟开始累积。累计达到或超过 $a$ 时，选择该扫描分钟的 close，只生成一个点；随后将累积量清零，盈余舍弃，再继续扫描。

过去方向从 $t$ 所在分钟开始累积，包含该分钟的增量。累计达到或超过 $a$ 时，选择扫描到的分钟之前一条实际记录的 close，再清零继续向过去扫描。例如从 09:35 向过去扫描，若 09:35 这一分钟的增量已达到 $a$，新增过去点取 09:34 的 close；训练侧末尾仍保留 09:35 的 close。休市两侧的“上一分钟”指上一条实际分钟记录。

准备时一次计算各分钟在两个方向上的下一采样位置，滚动采样可以重复使用这些位置。


In [6]:
@njit(cache=False)
def _reset_maps(cumulative, upper, a):
    length = len(cumulative)
    past = np.full(length, -1, dtype=np.int64)  # -1 表示没有可达的上一点
    future = np.full(length, -1, dtype=np.int64)
    past_cursor = -1
    future_cursor = 1

    # 逐分钟建立双向下一点映射；每次跳转都从该点重新累计。
    for origin in range(length):

        # 过去累计包含当前分钟；达标后取扫描分钟之前的实际记录 close。
        tolerance = upper[origin] - cumulative[origin]
        threshold = cumulative[origin] - a + tolerance
        while past_cursor + 1 < origin and cumulative[past_cursor + 1] <= threshold:
            past_cursor += 1
        past[origin] = past_cursor  # 该源分钟将作为过去点的 close

        # 未来严格从 origin 后开始，达到阈值时取该扫描分钟 close。
        future_cursor = max(future_cursor, origin + 1)
        threshold = cumulative[origin] + a
        while future_cursor < length and upper[future_cursor] < threshold:
            future_cursor += 1
        if future_cursor < length:
            future[origin] = future_cursor
    return past, future


def prepare_sampling_axis(axis_minute_df: pd.DataFrame, *, a: float, sampling_method: str) -> dict:
    """一次准备固定单位的累计轴；滚动分界复用同一结果。"""
    if not np.isfinite(a) or a <= 0:
        raise ValueError('采样单位 a 必须为有限正数')
    if sampling_method not in ('log_interpolate', 'minute_close_reset'):
        raise ValueError('sampling_method 必须为 log_interpolate 或 minute_close_reset')
    minutes = axis_minute_df.reset_index(drop=True)
    if minutes.empty:
        raise ValueError('采样输入没有实际分钟')
    increments = minutes['sampling_increment'].to_numpy(dtype=float, na_value=np.nan)
    if (~np.isfinite(increments) | (increments < 0)).any():
        raise ValueError('采样增量必须有限且非负')

    # 只累计实际分钟；午休、隔夜和跨交易日均不补行、不重置。
    cumulative = np.cumsum(increments)
    if not np.isfinite(cumulative).all():
        raise ValueError('累计采样增量溢出')

    # 只补偿机器舍入误差，防止恰好达标的刻度漏点。
    upper = cumulative + 4 * np.finfo(float).eps * np.maximum(cumulative, a)
    if (upper - cumulative >= a / 2).any():
        raise ValueError('采样单位小于累计轴可分辨精度')
    times = pd.DatetimeIndex(minutes['bar_at']).as_unit('ns')  # 统一分钟索引的时间精度
    prepared = dict(minutes=minutes, increments=increments, cumulative=cumulative,
                    upper=upper, a=float(a), sampling_method=sampling_method, times=times)

    # 清零方式一次准备跳转表，滚动时直接复用。
    if sampling_method == 'minute_close_reset':
        prepared['past_map'], prepared['future_map'] = _reset_maps(cumulative, upper, a)
    return prepared


def _anchor_indices(prepared_axis, t):
    times = prepared_axis['times']

    # 将单个或多个 t 统一为上海时区的分钟末端。
    anchors = pd.DatetimeIndex([t] if np.isscalar(t) or isinstance(t, datetime) else t)
    if anchors.tz is None:
        anchors = anchors.tz_localize('Asia/Shanghai')
    else:
        anchors = anchors.tz_convert('Asia/Shanghai')

    # 允许实际分钟末端，另允许首分钟起点作为全区间测试的分界。
    indices = times.get_indexer(anchors)
    at_start = anchors == times[0] - pd.Timedelta(minutes=1)  # 此特例的训练侧为空
    if ((indices < 0) & ~at_start).any():
        raise ValueError('t 必须是实际分钟末端，或首个实际分钟的起点（整个输入作为测试期）')
    if not anchors.is_monotonic_increasing or anchors.has_duplicates:  # 滚动游标要求 t 递增且不重复
        raise ValueError('批量 t 必须按时间递增且不重复')
    return anchors, indices


## 指定分界与滚动分界

<a id="fixed-and-rolling-boundaries"></a>

`generate_sampling_positions(prepared_axis, t=t, K=K)` 计算一个分界的两侧位置。普通分界 $t$ 必须是输入中已有的分钟末端：该分钟 close 已知，因此属于训练侧，未来从下一实际分钟开始。

两侧结果都按时间从早到晚排列。训练侧最后一个点固定为 $t$ 的 close，并计入 $K$；测试侧不重复这个点。`K` 为正整数时，每侧最多返回 $K$ 点，范围内不足时返回实际可得点数；`K=None` 返回该分界两侧在输入范围内的完整采样序列。返回表通过 `t` 和 `sample_side` 标明分界及训练／测试侧，通过 `source_minute_index`、`bar_at` 和 `interpolation_fraction` 定位源分钟及比例。

需要整个输入都作为测试期时，可将首个实际分钟的起点，即首行 `bar_at` 减一分钟，作为 $t$。此时训练侧为空，测试从首个实际分钟开始，仍须提供事先确定的正单位 $a$。

`iter_sampling_position_batches(prepared_axis, t=t_sequence, K=K, batch_size=512)` 接收按时间递增且不重复的分界序列，分批返回各分界两侧的位置；滚动模式必须指定正整数 $K$。每个分界使用相同的累计轴、单位与采样方式。插值定位复用随分界前进的游标，分钟末清零复用准备好的双向位置，以减少逐分钟重新扫描。


In [7]:
@njit(cache=False, parallel=True)
def _interpolated_positions(cumulative, upper, increments, a, origins, past_count, future_count):
    size = len(origins)
    past = np.full((past_count, size), -1, dtype=np.int64)  # -1 为不可达位置，打包时移除
    future = np.full((future_count, size), -1, dtype=np.int64)
    past_fraction = np.full((past_count, size), np.nan)
    future_fraction = np.full((future_count, size), np.nan)

    # 先把 t 的 close 放在训练侧末尾，这个点占用一个名额。
    for row in range(size):
        if origins[row] >= 0 and past_count:
            past[past_count - 1, row] = origins[row]
            past_fraction[past_count - 1, row] = 1.0

    # 按距离并行定位；同一距离下复用随 t 前进的游标。
    for lag in prange(1, max(past_count, future_count + 1)):
        for direction in range(2):
            if direction == 0 and lag >= past_count or direction == 1 and lag > future_count:
                continue
            cursor = 0
            initialized = False
            for row in range(size):
                origin = origins[row]
                if direction == 0 and origin < 0:
                    continue
                base = cumulative[origin] if origin >= 0 else 0.0  # 包含 t 所在分钟的增量

                # 目标刻度相对 t 向两侧等距推进，范围外的整点舍弃。
                target = base - lag * a if direction == 0 else base + lag * a
                tolerance = 4 * np.finfo(np.float64).eps * max(abs(base), a)
                if target < -tolerance or target > upper[-1]:
                    continue
                target = max(target, 0.0)

                # 只为首个分界搜索起点，后续分界沿累计轴向前移动。
                if not initialized:
                    cursor = np.searchsorted(upper, target)
                    initialized = True
                while cursor < len(upper) and upper[cursor] < target:
                    cursor += 1
                if cursor == len(upper):
                    continue

                # 零刻度取首分钟 open；其他刻度计算所在分钟及 open→close 比例。
                if target == 0.0:
                    cursor = 0
                    fraction = 0.0
                else:
                    previous = cumulative[cursor - 1] if cursor else 0.0
                    end_tolerance = upper[cursor] - cumulative[cursor]
                    if abs(target - cumulative[cursor]) <= end_tolerance:
                        fraction = 1.0  # 恰好落在分钟末端
                    elif increments[cursor] > 0:
                        fraction = (target - previous) / increments[cursor]
                    else:
                        continue

                # 过去点倒序填入，返回时两侧都按时间递增排列。
                if direction == 0:
                    past[past_count - 1 - lag, row] = cursor
                    past_fraction[past_count - 1 - lag, row] = fraction
                else:
                    future[lag - 1, row] = cursor
                    future_fraction[lag - 1, row] = fraction
    return past.T.copy(), past_fraction.T.copy(), future.T.copy(), future_fraction.T.copy()  # 按分界组织连续内存


@njit(cache=False, parallel=True)
def _reset_positions(past_map, future_map, upper, a, origins, count):
    past = np.full((len(origins), count), -1, dtype=np.int64)
    future = np.full((len(origins), count), -1, dtype=np.int64)

    # 各分界独立沿准备好的清零映射跳转。
    for row in prange(len(origins)):
        source = origins[row]
        if source >= 0:
            past[row, count - 1] = source  # t 的 close 计入 K
            for column in range(count - 2, -1, -1):
                source = past_map[source]
                if source < 0:
                    break
                past[row, column] = source
        source = origins[row]

        # 未来不保留 t；全区间测试从第一个达到 a 的分钟开始。
        for column in range(count):
            source = (np.searchsorted(upper, a) if source < 0 and column == 0
                      else future_map[source])
            if source < 0 or source >= len(upper):
                break
            future[row, column] = source
    return past, np.ones(past.shape), future, np.ones(future.shape)  # 清零模式全部取 close，比例均为 1


def _pack_positions(prepared_axis, anchors, arrays, *, first_sample_id=1):
    past, past_fraction, future, future_fraction = arrays
    sources = np.concatenate((past, future), axis=1)
    fractions = np.concatenate((past_fraction, future_fraction), axis=1)

    # 移除不可达点，保留其余点的位置和比例，不补足 K。
    valid = sources >= 0
    anchor_rows, columns = np.nonzero(valid)
    selected = sources[valid]
    fraction = fractions[valid]
    if ((fraction < 0) | (fraction > 1) | ~np.isfinite(fraction)).any():
        raise ValueError('采样插值比例超出 [0,1]')

    # 训练、测试各自从最早点编号，sample_id 在输出中连续递增。
    past_number = np.cumsum(past >= 0, axis=1)
    future_number = np.cumsum(future >= 0, axis=1)
    numbers = np.concatenate((past_number, future_number), axis=1)[valid]
    result = prepared_axis['minutes'].iloc[selected][['trading_date', 'bar_at', 'main_contract_code']].reset_index(drop=True)
    result['sample_id'] = np.arange(first_sample_id, first_sample_id + len(result), dtype=np.int64)
    result['t'] = anchors.take(anchor_rows)  # 标明该点所属的采样分界
    result['sample_side'] = np.where(columns < past.shape[1], 'train', 'test')  # 前半矩阵为训练，后半为测试
    result['sample_number'] = numbers
    result['source_minute_index'] = selected
    result['interpolation_fraction'] = fraction

    # 用源分钟末端减去剩余分钟比例，得到分钟内时间。
    result['sample_at'] = pd.to_datetime(result['bar_at']) - pd.to_timedelta(
        np.rint((1 - fraction) * 60_000_000).astype(np.int64), unit='us')
    return pa.Table.from_pandas(result, schema=SAMPLING_POSITION_SCHEMA, preserve_index=False).to_pandas(types_mapper=pd.ArrowDtype)


def generate_sampling_positions(prepared_axis: dict, *, t, K: int | None = None) -> pd.DataFrame:
    """从一个明确分界 t 生成过去／未来位置；K=None 取允许范围内全部完整点。"""
    anchors, origins = _anchor_indices(prepared_axis, t)
    if len(origins) != 1:
        raise ValueError('单次采样需要一个 t；多时点使用 iter_sampling_position_batches')

    # 有限 K 只计算每侧需要的点；不足时由打包环节返回实际结果。
    if K is not None:
        if isinstance(K, bool) or not isinstance(K, (int, np.integer)) or K <= 0:
            raise ValueError('K 必须是正整数或 None')
        arrays = _limited_positions(prepared_axis, origins, int(K))

    # 完整插值序列：按可用累计距离确定两侧整点数。
    elif prepared_axis['sampling_method'] == 'log_interpolate':
        origin = origins[0]
        cumulative = prepared_axis['cumulative']
        a = prepared_axis['a']
        base = cumulative[origin] if origin >= 0 else 0.0
        tolerance = 4 * np.finfo(float).eps * max(base, a)
        past_count = int(np.floor((base + tolerance) / a)) + 1 if origin >= 0 else 0  # +1 保留 t 的 close
        future_count = int(np.floor((cumulative[-1] - base + tolerance) / a))  # 尾部不足一个 a 的量舍弃
        arrays = _interpolated_positions(cumulative, prepared_axis['upper'], prepared_axis['increments'],
                                        a, origins, past_count, future_count)
    else:
        origin = int(origins[0])

        # 完整清零序列：沿双向映射跳转到输入边界。
        past, future = [], []
        source = origin
        while source >= 0:
            past.append(source)
            source = int(prepared_axis['past_map'][source])
        source = origin
        while True:
            source = (int(np.searchsorted(prepared_axis['upper'], prepared_axis['a']))
                      if source < 0 and not future else int(prepared_axis['future_map'][source]))
            if source < 0 or source >= len(prepared_axis['minutes']):
                break
            future.append(source)
        past = np.asarray(past[::-1], dtype=np.int64).reshape(1, -1)  # 反转过去扫描结果，按时间递增返回
        future = np.asarray(future, dtype=np.int64).reshape(1, -1)
        arrays = past, np.ones(past.shape), future, np.ones(future.shape)
    return _pack_positions(prepared_axis, anchors, arrays)


def _limited_positions(prepared_axis, origins, K):

    # 选择同一分界规则，两侧容量均为 K。
    if prepared_axis['sampling_method'] == 'log_interpolate':
        return _interpolated_positions(prepared_axis['cumulative'], prepared_axis['upper'],
                                       prepared_axis['increments'], prepared_axis['a'], origins, K, K)
    return _reset_positions(prepared_axis['past_map'], prepared_axis['future_map'],
                            prepared_axis['upper'], prepared_axis['a'], origins, K)


def iter_sampling_position_batches(prepared_axis: dict, *, t, K: int, batch_size: int = 512):
    """按给定的递增 t 序列分批滚动；每侧最多 K 点，不补不足的点。"""
    if isinstance(K, bool) or not isinstance(K, (int, np.integer)) or K <= 0:
        raise ValueError('K 必须是正整数')
    if isinstance(batch_size, bool) or not isinstance(batch_size, (int, np.integer)) or batch_size <= 0:
        raise ValueError('batch_size 必须是正整数')
    anchors, origins = _anchor_indices(prepared_axis, t)
    next_id = 1

    # 按分界批次生成位置，跨批次保持采样编号连续。
    for first in range(0, len(origins), batch_size):
        stop = min(first + batch_size, len(origins))
        arrays = _limited_positions(prepared_axis, origins[first:stop], int(K))
        positions = _pack_positions(prepared_axis, anchors[first:stop], arrays, first_sample_id=next_id)
        next_id += len(positions)  # 下一批从这里接续编号
        yield positions


## 合成分钟位置 demo

<a id="synthetic-position-demo"></a>

下面六个连续分钟的增量为 `[8, 15, 0, 7, 12, 0]`，单位 $a=10$。固定分界取第四分钟 09:34；滚动分界依次取六个分钟末端，每侧最多 $K=4$ 点。

本节直接给定增量与单位，用于查看两个方向、零增量和端点对应的位置。将 `demo_sampling_method` 设为 `log_interpolate` 或 `minute_close_reset` 可查看两种规则的结果。固定分界返回完整序列，滚动结果预览前 20 行。


In [8]:
# 给定六分钟增量，直接核对方向、零增量和端点位置。
demo_times = pd.date_range('2020-06-01 09:31', periods=6, freq='min', tz='Asia/Shanghai')
demo_axis = pd.DataFrame({'trading_date': demo_times.date, 'bar_at': demo_times,
                         'main_contract_code': 'demo', 'sampling_increment': [8., 15., 0., 7., 12., 0.]})
demo_a = 10.0  # 直接给定的单位
demo_K = 4  # 每侧最多 4 点
demo_t = demo_times[3]  # 第四分钟 09:34 的末端
demo_sampling_method = 'log_interpolate'
demo_prepared_axis = prepare_sampling_axis(demo_axis, a=demo_a, sampling_method=demo_sampling_method)

# 同一准备结果分别用于固定分界完整序列和逐分钟滚动。
fixed_positions = generate_sampling_positions(demo_prepared_axis, t=demo_t, K=None)
rolling_positions = pd.concat(iter_sampling_position_batches(demo_prepared_axis, t=demo_times, K=demo_K), ignore_index=True)
display(fixed_positions.rename(columns=POSITION_COLUMN_LABELS))
display(rolling_positions.head(20).rename(columns=POSITION_COLUMN_LABELS))


,sample_id（采样编号）,t（训练期末端）,sample_side（采样所属区间）,sample_number（区间内采样序号）,source_minute_index（源实际分钟位置）,sample_at（分钟内采样时间）,trading_date（bar 归属的交易日，而非其自然日期）,bar_at（一分钟 bar 的结束时刻）,main_contract_code（当分钟主力合约）,interpolation_fraction（源分钟内插值比例）
0,1,2020-06-01 09:34:00+08:00,train,1,0,2020-06-01 09:30:00+08:00,2020-06-01,2020-06-01 09:31:00+08:00,demo,0.0
1,2,2020-06-01 09:34:00+08:00,train,2,1,2020-06-01 09:31:08+08:00,2020-06-01,2020-06-01 09:32:00+08:00,demo,0.133333
2,3,2020-06-01 09:34:00+08:00,train,3,1,2020-06-01 09:31:48+08:00,2020-06-01,2020-06-01 09:32:00+08:00,demo,0.8
3,4,2020-06-01 09:34:00+08:00,train,4,3,2020-06-01 09:34:00+08:00,2020-06-01,2020-06-01 09:34:00+08:00,demo,1.0
4,5,2020-06-01 09:34:00+08:00,test,1,4,2020-06-01 09:34:50+08:00,2020-06-01,2020-06-01 09:35:00+08:00,demo,0.833333


,sample_id（采样编号）,t（训练期末端）,sample_side（采样所属区间）,sample_number（区间内采样序号）,source_minute_index（源实际分钟位置）,sample_at（分钟内采样时间）,trading_date（bar 归属的交易日，而非其自然日期）,bar_at（一分钟 bar 的结束时刻）,main_contract_code（当分钟主力合约）,interpolation_fraction（源分钟内插值比例）
0,1,2020-06-01 09:31:00+08:00,train,1,0,2020-06-01 09:31:00+08:00,2020-06-01,2020-06-01 09:31:00+08:00,demo,1.0
1,2,2020-06-01 09:31:00+08:00,test,1,1,2020-06-01 09:31:40+08:00,2020-06-01,2020-06-01 09:32:00+08:00,demo,0.666667
2,3,2020-06-01 09:31:00+08:00,test,2,3,2020-06-01 09:33:42.857143+08:00,2020-06-01,2020-06-01 09:34:00+08:00,demo,0.714286
3,4,2020-06-01 09:31:00+08:00,test,3,4,2020-06-01 09:34:40+08:00,2020-06-01,2020-06-01 09:35:00+08:00,demo,0.666667
4,5,2020-06-01 09:32:00+08:00,train,1,0,2020-06-01 09:30:22.500000+08:00,2020-06-01,2020-06-01 09:31:00+08:00,demo,0.375
5,6,2020-06-01 09:32:00+08:00,train,2,1,2020-06-01 09:31:20+08:00,2020-06-01,2020-06-01 09:32:00+08:00,demo,0.333333
6,7,2020-06-01 09:32:00+08:00,train,3,1,2020-06-01 09:32:00+08:00,2020-06-01,2020-06-01 09:32:00+08:00,demo,1.0
7,8,2020-06-01 09:32:00+08:00,test,1,4,2020-06-01 09:34:15+08:00,2020-06-01,2020-06-01 09:35:00+08:00,demo,0.25
8,9,2020-06-01 09:33:00+08:00,train,1,0,2020-06-01 09:30:22.500000+08:00,2020-06-01,2020-06-01 09:31:00+08:00,demo,0.375
9,10,2020-06-01 09:33:00+08:00,train,2,1,2020-06-01 09:31:20+08:00,2020-06-01,2020-06-01 09:32:00+08:00,demo,0.333333


## 真实分钟数据的位置 demo

<a id="actual-position-demo"></a>

本节读取主力连续复权数据，每次选择一个品种、$N$、轴、成交额范围和采样方式，校准单位后保存一个固定分界的位置文件。品种可选 CU、RB，$N$ 可选 10、15、30、60；默认示范 RB、$N=10$、全部合约成交额及分钟内插值。

请求范围为 2010-01-01 至 2026-09-30，实际起止分钟由输入文件决定。初始校准使用 `trading_date < 2025-01-01` 的全部实际分钟，$t$ 取该训练期最后一个实际分钟末端，`K=None`。文件名分别写明训练范围 `train20100101-20241231` 与测试范围 `test20250101-20260930`。

输入文件通过 `demo_upstream_path` 的完整名称选择。运行后展示校准结果、分钟增量和采样位置，并将位置保存到本方法的 `c01_SamplingPositions_DemoData/`，使用 `c01_SamplingPositions_..._fixed_demo.parquet` 前缀。各合约价格及主力序列图由[采样执行 Notebook](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c02_SamplingExecution.ipynb)生成。


`demo_reuse_existing=True` 时，已有配置的上游身份、参数、生成定义、数值环境和内容摘要一致，就读取保存的位置及校准值，跳过分钟轴生成、校准与采样。缺失或条件变化时才计算；文件损坏明确报错。设为 `False` 可显式重算。


In [ ]:
from pathlib import Path
from datetime import date, datetime
import numba
import pyarrow.parquet as pq
from R04_Research.a01_Methods.b03_Sampling.c02_SamplingExecution import (
    iter_adjusted_minute_batches, verify_adjusted_demo, save_sampling_demo, sampling_demo_identity,
    EXECUTION_COLUMN_LABELS, read_reusable_sampling_identity,
)

# 选择一个业务配置，修改这些参数后运行对应 demo。
demo_underlying_code = 'RB'  # 品种：CU 或 RB
demo_N = 10  # 候选分钟数：10、15、30、60
demo_axis_name = 'money'  # time、money 或 gk_money
demo_quantity_scope = 'all_contracts'  # 全部合约合计；也支持 main_contract
demo_sampling_method = 'log_interpolate'  # 分钟内插值；也支持 minute_close_reset
demo_test_start_date = date(2025, 1, 1)  # 按 trading_date 划分初始校准训练期
demo_range_token = 'train20100101-20241231_test20250101-20260930'
demo_reuse_existing = True  # 条件一致时复用；False 显式重算
demo_threads = 8  # 数值环境中的并行线程数
numba.set_num_threads(min(demo_threads, numba.config.NUMBA_NUM_THREADS))

# 用完整文件名指定复权输入和所属输出，训练／测试范围分别写入名称。
demo_upstream_notebook = candidate_root / 'R04_Research/a01_Methods/b02_MainContinuousAdjustment/c01_MainContinuousAdjustment.ipynb'
demo_upstream_path = (
    demo_upstream_notebook.parent / 'c01_MainContinuousAdjustment_DemoData'
    / f'c01_MainContinuousAdjustment_{demo_underlying_code}_{demo_range_token}_ratio1.10_log_common_demo.parquet'
)
demo_method_notebook = candidate_root / 'R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions.ipynb'
demo_output_path = (
    demo_method_notebook.parent / 'c01_SamplingPositions_DemoData'
    / f'{demo_method_notebook.stem}_{demo_underlying_code}_{demo_range_token}_'
      f'N{demo_N}_{demo_axis_name}_{demo_quantity_scope}_{demo_sampling_method}_fixed_demo.parquet'
)

# 已有结果匹配时直接读取，在分钟轴、GK、校准和采样之前分支。
demo_requested_parameters = dict(N=demo_N, axis=demo_axis_name, quantity_scope=demo_quantity_scope,
    sampling_method=demo_sampling_method, execution_mode='fixed', K=None)
demo_existing_identity = read_reusable_sampling_identity(demo_output_path, project_root=candidate_root,
    producer_notebook=demo_method_notebook, upstream_path=demo_upstream_path,
    underlying_code=demo_underlying_code, test_start_date=demo_test_start_date,
    parameters=demo_requested_parameters, reuse_existing=demo_reuse_existing)
demo_axis_minutes = None
if demo_existing_identity is not None:
    demo_calibration = demo_existing_identity['calibration']
    demo_positions = pq.read_table(demo_output_path).to_pandas(types_mapper=pd.ArrowDtype)
    demo_save = {'identity': demo_existing_identity, 'preserved_existing': True}
    print('复用已有位置数据；跳过分钟轴、初始校准与采样。')
else:
    # 核对固定复权输入，再准备实际分钟采样轴。
    demo_upstream_identity = verify_adjusted_demo(demo_upstream_path, producer_notebook=demo_upstream_notebook,
        project_root=candidate_root, underlying_code=demo_underlying_code, test_start_date=demo_test_start_date)
    demo_axis_parts = [generate_sampling_column(batch, axis=demo_axis_name, quantity_scope=demo_quantity_scope)
                       for batch in iter_adjusted_minute_batches(demo_upstream_path)]
    demo_axis_minutes = pd.concat(demo_axis_parts, ignore_index=True)
    del demo_axis_parts

    # 只按初始训练期校准 a，再准备可复用的累计轴。
    demo_calibration = calibrate_sampling_unit(demo_axis_minutes, N=demo_N, test_start_date=demo_test_start_date)
    demo_prepared_axis = prepare_sampling_axis(demo_axis_minutes, a=demo_calibration['a'], sampling_method=demo_sampling_method)

    # 固定分界取训练末实际分钟，返回两侧完整序列。
    demo_t = demo_axis_minutes.loc[demo_axis_minutes['trading_date'] < demo_test_start_date, 'bar_at'].iloc[-1]
    demo_positions = generate_sampling_positions(demo_prepared_axis, t=demo_t, K=None)  # K=None，不限制每侧点数

    # 记录这个配置的生成条件，保存后复读验收。
    demo_identity = sampling_demo_identity(project_root=candidate_root, producer_notebook=demo_method_notebook,
        upstream_identity=demo_upstream_identity, calibration=demo_calibration, scope=demo_upstream_identity['scope'],
        parameters=dict(N=demo_N, axis=demo_axis_name, quantity_scope=demo_quantity_scope,
                        sampling_method=demo_sampling_method, execution_mode='fixed', t=str(demo_t), K=None))
    demo_save = save_sampling_demo(demo_positions, demo_path=demo_output_path,
                                   demo_identity=demo_identity, schema=SAMPLING_POSITION_SCHEMA)

# 只给展示副本加中文表头，计算和文件仍保留原字段。
display(pd.DataFrame([demo_calibration]).rename(columns=EXECUTION_COLUMN_LABELS))
if demo_axis_minutes is not None:
    display(demo_axis_minutes.head().rename(columns=POSITION_COLUMN_LABELS))
display(demo_positions.head().rename(columns=POSITION_COLUMN_LABELS))

## 48 组位置数据 demo

<a id="saved-位置-demos"></a>

各配置独立生成，使用固定分界和完整序列（`K=None`）。时间轴累计实际分钟；成交额和 GK×成交额轴使用全部合约成交额。训练交易日为 2010-01-01 至 2024-12-31，测试交易日为 2025-01-01 至 2026-09-30；实际分钟由 b02 输入决定。

| 品种 | N | 采样轴 | 采样方式 | 位置文件 |
| --- | ---: | --- | --- | --- |
| CU | 10 | 实际分钟 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N10_time_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 10 | 实际分钟 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N10_time_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 10 | 成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N10_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 10 | 成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N10_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 10 | GK×成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N10_gk_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 10 | GK×成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N10_gk_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 15 | 实际分钟 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N15_time_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 15 | 实际分钟 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N15_time_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 15 | 成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N15_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 15 | 成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N15_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 15 | GK×成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N15_gk_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 15 | GK×成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N15_gk_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 30 | 实际分钟 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N30_time_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 30 | 实际分钟 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N30_time_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 30 | 成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N30_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 30 | 成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N30_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 30 | GK×成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N30_gk_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 30 | GK×成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N30_gk_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 60 | 实际分钟 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N60_time_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 60 | 实际分钟 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N60_time_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 60 | 成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N60_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 60 | 成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N60_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| CU | 60 | GK×成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N60_gk_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| CU | 60 | GK×成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_CU_train20100101-20241231_test20250101-20260930_N60_gk_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 10 | 实际分钟 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N10_time_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 10 | 实际分钟 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N10_time_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 10 | 成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N10_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 10 | 成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N10_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 10 | GK×成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N10_gk_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 10 | GK×成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N10_gk_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 15 | 实际分钟 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N15_time_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 15 | 实际分钟 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N15_time_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 15 | 成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N15_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 15 | 成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N15_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 15 | GK×成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N15_gk_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 15 | GK×成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N15_gk_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 30 | 实际分钟 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N30_time_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 30 | 实际分钟 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N30_time_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 30 | 成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N30_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 30 | 成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N30_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 30 | GK×成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N30_gk_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 30 | GK×成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N30_gk_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 60 | 实际分钟 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N60_time_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 60 | 实际分钟 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N60_time_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 60 | 成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N60_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 60 | 成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N60_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
| RB | 60 | GK×成交额 | 分钟内插值 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N60_gk_money_all_contracts_log_interpolate_fixed_demo.parquet) |
| RB | 60 | GK×成交额 | 分钟末清零 | [位置 Parquet](E:/Latitude_Analytics_v2/R04_Research/a01_Methods/b03_Sampling/c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N60_gk_money_all_contracts_minute_close_reset_fixed_demo.parquet) |
